In [1]:
#!pip install pybricks
#!pip install network

In [2]:
#!/usr/bin/env pybricks-micropython

####################################################################################
# Bloc d'importation des bibliothèques nécessaires
####################################################################################

# from pybricks.hubs import EV3Brick
from pybricks.ev3devices import (Motor, TouchSensor, ColorSensor,
                                 InfraredSensor, UltrasonicSensor, GyroSensor)
from pybricks.parameters import Port, Stop, Direction, Button, Color
from pybricks.tools import wait, StopWatch, DataLog
from pybricks.robotics import DriveBase
from pybricks.media.ev3dev import SoundFile, ImageFile

import network
####################################################################################
# Classes de controle des moteurs
####################################################################################

In [ ]:
# Version of code is : 2

#!/usr/bin/env pybricks-micropython

####################################################################################
# Bloc d'importation des bibliothèques nécessaires
####################################################################################

from pybricks.hubs import EV3Brick
from pybricks.ev3devices import (Motor, TouchSensor, ColorSensor,
                                 InfraredSensor, UltrasonicSensor, GyroSensor)
from pybricks.parameters import Port, Stop, Direction, Button, Color
from pybricks.tools import wait, StopWatch, DataLog
from pybricks.robotics import DriveBase
from pybricks.media.ev3dev import SoundFile, ImageFile

####################################################################################
# Classe de contrôle du robot
####################################################################################

class ControlRobot:
    def __init__(self, left_motor_port=Port.B, right_motor_port=Port.C):
        self.left_motor = Motor(left_motor_port)
        self.right_motor = Motor(right_motor_port)
        self.robot = DriveBase(self.left_motor, self.right_motor, wheel_diameter=55.5, axle_track=104)

    def rotate_with_speed(self, speed=100, turn_rate=0):
        self.robot.drive(speed, turn_rate)

    def move_forward(self, speed=100):
        self.robot.drive(speed, 0)

    def move_backward(self, speed=100):
        self.robot.drive(-speed, 0)

    def init_distance_mesurement(self):
        self.robot.reset()

    def get_distance_mesure(self):
        return self.robot.distance()
    
    def emergency_stop(self):
        self.robot.stop()


class SensRobot:
    def __init__(self, ultrasonic_sensor_port=Port.S4, color_sensor_port=Port.S3):
        self.color_sensor = ColorSensor(color_sensor_port)
        self.ultrasonic_sensor = UltrasonicSensor(ultrasonic_sensor_port)

    def read_sensors(self):
        color = self.color_sensor.color()
        distance = self.ultrasonic_sensor.distance()
        reflection = self.color_sensor.reflection()
        return {"color": color, "distance": distance, "reflection": reflection}

class PrintInfo:

    def __init__(self):
        pass

    def print_sensor_info(self,sensor_data):
        print(sensor_data)

    def print_on_lcd(self, sensor_data):
        ev3 = EV3Brick()
        ev3.screen.clear()
        ev3.screen.draw_text(sensor_data)

# class RobotNetwork:
#     def __init__(self, ssid, password):
#         self.wlan = network.WLAN(network.STA_IF)
#         self.wlan.active(True)
#         self.wlan.connect(ssid, password)
#         while not self.wlan.isconnected():
#             pass
#         print('network config:', self.wlan.ifconfig())

class RobotLoging:
    def __init__(self, *columns):
        self.log = DataLog(*columns)

    def log_data(self, *data):
        self.log.log(*data)

    def save_log(self, filename):
        self.log.save(filename)

########################################################################################
#Implementation du suivis de ligne
#########################################################################################


# Initialiser le chronomètre
stopwatch = StopWatch()

# Démarrer le chronomètre
stopwatch.reset()
stopwatch.resume()

execution_time = 100000  # 1000 ms = 1 seconde
distance_to_travel = 1000  # Distance in mm


##############################################################################

# Initialize the motors.
left_motor = (Port.B)
right_motor = (Port.C)

color_sensor = Port.S3
ultrasonic_sensor = Port.S2

control_robot = ControlRobot(right_motor_port=right_motor, left_motor_port=left_motor)
sens_robot = SensRobot(ultrasonic_sensor_port=ultrasonic_sensor, color_sensor_port=color_sensor)
print_info = PrintInfo()

##############################################################################

# Set statics values

white_line_reflection =  60  # Adjust this value based on your environment
black_line_reflection = 10   # Adjust this value based on your environment
middle_reflection = 30 #(white_line_reflection + black_line_reflection) / 2
base_speed = 300
actual_distance = 0

##############################################################################
# Bang-bang line following algorithm
##############################################################################

# control_robot.init_distance_mesurement()

# while actual_distance < distance_to_travel:  # Loop forever
#     # Optionnel : afficher le temps écoulé
# #    print(f"Temps écoulé : {stopwatch.time()} ms")

#     print(actual_distance)

#     control_robot.move_forward(speed=base_speed)

#     sensor_data = sens_robot.read_sensors()
#     reflection = sensor_data['reflection']
#     error = reflection - middle_reflection

#     control_robot.rotate_with_speed(turn_rate = error, speed=(base_speed))

#     log.log(stopwatch.time(), error, control_robot.get_distance_mesure())

#     wait(10)  # Small delay to avoid overwhelming the CPU

#     actual_distance = control_robot.get_distance_mesure()

# #print(f"Temps final : {stopwatch.time()} ms")

##############################################################################
# PID line following algorithm
##############################################################################

# Initialiser les constantes PID
Kp = 0.2  # Proportional gain
Ki = 1.0 # Integral gain
Kd = 0.8  # Derivative gain
sum_error = 0
last_error = 0
wait_time = 100  #Division par zero

param = (Kp, Ki, Kd)
log = DataLog("temps", "erreur", "distance", "Kp", "Ki", "Kd")

control_robot.init_distance_mesurement()

while actual_distance < distance_to_travel:  # Loop forever
    # Optionnel : afficher le temps écoulé
#    print(f"Temps écoulé : {stopwatch.time()} ms")

    print(actual_distance)

    control_robot.move_forward(speed=base_speed)

    sensor_data = sens_robot.read_sensors()
    reflection = sensor_data['reflection']
    error = reflection - middle_reflection
    sum_error += error
    command = error * Kp + sum_error * Ki #+ (error - last_error)/wait_time * Kd
    last_error = error

    control_robot.rotate_with_speed(turn_rate = command, speed=(base_speed))

    log.log(stopwatch.time(), error, control_robot.get_distance_mesure())

    wait(100)  # Small delay to avoid overwhelming the CPU

    actual_distance = control_robot.get_distance_mesure()

#print(f"Temps final : {stopwatch.time()} ms")

In [ ]:
#Version of code is : latest 


#!/usr/bin/env pybricks-micropython

####################################################################################
# Bloc d'importation des biblio nécessaires
####################################################################################

from pybricks.hubs import EV3Brick
from pybricks.ev3devices import Motor, TouchSensor, ColorSensor,InfraredSensor, UltrasonicSensor, GyroSensor
from pybricks.parameters import Port, Stop, Direction, Button, Color
from pybricks.tools import wait, StopWatch, DataLog
from pybricks.robotics import DriveBase
#from pybricks.media.ev3dev import SoundFile, ImageFile

from pybricks.media.ev3dev import Screen

####################################################################################
# Classe de contrôle du robot
####################################################################################

class ControlRobot:
    def __init__(self, left_motor_port=Port.B, right_motor_port=Port.C):
        self.left_motor = Motor(left_motor_port)
        self.right_motor = Motor(right_motor_port)
        self.robot = DriveBase(self.left_motor, self.right_motor, wheel_diameter=55.5, axle_track=104)

    def rotate_with_speed(self, speed=100, turn_rate=0):
        self.robot.drive(speed, turn_rate)

    def move_forward(self, speed=100):
        self.robot.drive(speed, 0)

    def init_distance_mesurement(self):
        self.robot.reset()

    def get_distance_mesure(self):
        return self.robot.distance()
    
    def emergency_stop(self):
        self.robot.stop()

# Capteurs du robot

class SensRobot:
    def __init__(self, ultrasonic_sensor_port=Port.S4, color_sensor_port=Port.S3):
        self.color_sensor = ColorSensor(color_sensor_port)
        self.ultrasonic_sensor = UltrasonicSensor(ultrasonic_sensor_port)

    def read_sensors(self):
        color = self.color_sensor.color()
        distance = self.ultrasonic_sensor.distance()
        reflection = self.color_sensor.reflection()
        return {"color": color, "distance": distance, "reflection": reflection}

# Affichage des informations

class PrintInfo:

    def __init__(self):
        self.screen = Screen()

# Test Ajout de str pour fixer l'erreur d'affichage non gestion du dico
    def print_on_lcd(self, sensor_data):
        self.screen.clear()
        self.screen.draw_text(0, 0, f"Distance: {str(sensor_data['distance'])} mm")
        self.screen.draw_text(0, 20, f"Couleur: {str(sensor_data['color'])}")
        self.screen.draw_text(0, 40, f"Réflexion: {str(sensor_data['reflection'])}")

# TODO : A gerer apres Socket communication
# class RobotNetwork:
#     def __init__(self, ssid, password):
#         self.wlan = network.WLAN(network.STA_IF)
#         self.wlan.active(True)
#         self.wlan.connect(ssid, password)
#         while not self.wlan.isconnected():
#             pass
#         print('network config:', self.wlan.ifconfig())


class RobotLoging:
    def __init__(self, *columns):
        self.log = DataLog(*columns)

    def log_data(self, *data):
    safe_data = []
    for value in data:
        if value is None:
            safe_data.append("N/A")
        else:
            safe_data.append(str(value))
    self.log.log(*safe_data)


########################################################################################
#Implementation du suivis de ligne
#########################################################################################

ev3 = EV3Brick()
# Initialisation des moteurs
left_motor = (Port.B)
right_motor = (Port.C)

color_sensor = Port.S3
ultrasonic_sensor = Port.S2

control_robot = ControlRobot(right_motor_port=right_motor, left_motor_port=left_motor)
sens_robot = SensRobot(ultrasonic_sensor_port=ultrasonic_sensor, color_sensor_port=color_sensor)
print_info = PrintInfo()

# Les bornes
MAX_TURN = 100  # valeur max pour turn_rate
MIN_TURN = -100 # valeur min pour turn_rate

stopwatch = StopWatch()
stopwatch.reset()
stopwatch.resume()

##############################################################################
# Paramètres du suivi de ligne
##############################################################################

distance_to_travel = 1000  # Distance in mm : 1 m

white_line_reflection =  60 # Estimation apres calibration
black_line_reflection = 10  # Estimation apres calibration
middle_reflection = 30 #(white_line_reflection + black_line_reflection) / 2 mais par calibration on prend 30
base_speed = 300
actual_distance = 0

##############################################################################
# Bang-bang line following algorithm
##############################################################################

# control_robot.init_distance_mesurement()

# log = RobotLoging("temps", "erreur", "distance")

# while actual_distance < distance_to_travel:  # Loop forever
#     sensor_data = sens_robot.read_sensors()
#     reflection = sensor_data["reflection"]
#     distance = sensor_data["distance"]
#     color = sensor_data["color"]


#     # Affichage LCD
#     print_info.print_on_lcd(sensor_data)

#     # Bizzatement instabilité quand on ote ce bloc
#     control_robot.move_forward(speed=base_speed)

#     error = reflection - middle_reflection

# #    control_robot.rotate_with_speed(turn_rate = error, speed=(base_speed))


#     # Bang-Bang
#     if reflection > middle_reflection:
#         turn_rate = 100  # Trop clair → tourner à droite
#     else:
#         turn_rate = -100  # Trop sombre → tourner à gauche

#     bang_command = max(MIN_TURN, min(MAX_TURN, turn_rate))

#     control_robot.rotate_with_speed(turn_rate=bang_command, speed=base_speed)

#     log.log(stopwatch.time(), error, control_robot.get_distance_mesure())

#     actual_distance = control_robot.get_distance_mesure()
#     wait(10)  # Plus stable avec 10ms

#print(f"Temps final : {stopwatch.time()} ms")

##############################################################################
# PID line following algorithm
##############################################################################

# Initialiser PID
Kp, Ki, Kd = 1, 0.2, 0.1
sum_error, last_error = 0, 0
last_time = stopwatch.time() 
control_robot.init_distance_mesurement()

log = RobotLoging("temps", "erreur", "distance", "Kp", "Ki", "Kd")

while actual_distance < distance_to_travel:
    sensor_data = sens_robot.read_sensors()
    reflection = sensor_data["reflection"]
    distance = sensor_data["distance"]
    color = sensor_data["color"]

    # Affichage LCD
    print_info.print_on_lcd(sensor_data)

    # Calcul erreur
    error = reflection - middle_reflection

    # Calcul dt
    current_time = stopwatch.time()
    dt = (current_time - last_time) / 1000  # ms -> s

    #Division par zero donc : 
    dt = max(dt, 0.00000001)  # Empêcher dt d'être zéro

    # Calcul PID
    sum_error += error * dt                  # intégrale pondérée par le temps
    derivative = (error - last_error) / dt
    command = Kp * error + Ki * sum_error + Kd * derivative

    command = max(MIN_TURN, min(MAX_TURN, command))

    # la commande
    control_robot.rotate_with_speed(turn_rate=command, speed=base_speed)

    # Log
    log.log_data(current_time, error, distance, Kp, Ki, Kd)

    # Mise à jour pour prochaine itération
    last_error = error
    last_time = current_time
    actual_distance = control_robot.get_distance_mesure()

    wait(100)

#Fin
control_robot.emergency_stop()


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# Liste de fichiers (séparés par espace dans ta variable graph)
base_dir = "TP1/Logs/TP1"  # adapte selon où sont vraiment tes CSV

files = [os.path.join(base_dir, os.path.basename(f)) for f in files]

# Créer une figure avec 4 sous-graphiques
fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=("Reflection vs Time",
                                    "Error vs Time",
                                    "Command vs Time",
                                    "Distance vs Time"))

# Boucle sur les fichiers
for file_path in files:
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()

    # Nom court pour la légende
    label = os.path.basename(file_path).replace(".csv", "")

    # 1. Reflection
    fig.add_trace(go.Scatter(x=df["Time in ms"], y=df["Reflection"],
                             mode="lines", name=f"Reflection - {label}"),
                  row=1, col=1)

    # 2. Error
    fig.add_trace(go.Scatter(x=df["Time in ms"], y=df["Error"],
                             mode="lines", name=f"Error - {label}"),
                  row=2, col=1)

    # 3. Command
    fig.add_trace(go.Scatter(x=df["Time in ms"], y=df["Command"],
                             mode="lines", name=f"Command - {label}"),
                  row=3, col=1)

    # 4. Distance
    fig.add_trace(go.Scatter(x=df["Time in ms"], y=df["Distance in mm"],
                             mode="lines", name=f"Distance - {label}"),
                  row=4, col=1)

# Layout
fig.update_layout(height=1000, width=1200,
                  title_text="Line Follower Bang-Bang Control Logs (Comparaison multi-fichiers)",
                  showlegend=True)

# Légendes des axes
fig.update_xaxes(title_text="Time (ms)", row=4, col=1)
fig.update_yaxes(title_text="Reflection", row=1, col=1)
fig.update_yaxes(title_text="Error", row=2, col=1)
fig.update_yaxes(title_text="Command", row=3, col=1)
fig.update_yaxes(title_text="Distance (mm)", row=4, col=1)

fig.show()


FileNotFoundError: [Errno 2] No such file or directory: 'TP1/TP1/Logs/TP1/line_follower_version_bang_bang_26_09_2025_speed50correction2_2025_09_26_10_32_13_397247.csv'